# KryonImage 3.0M On-Device Image Generation Model Trainer
### Target Hardware: ESP32-S3 (e.g. N16R8 Octal PSRAM) running KryonOS v3.0.0
### Datasets:
- **Primary**: [Kaggle: Pokemon Sprite Images with Design Labels](https://www.kaggle.com/datasets/yehongjiang/pokemon-sprites-images) (`yehongjiang/pokemon-sprites-images`)
- **Hugging Face Mirror**: `reach-vb/pokemon-blip-captions` / `keremberke/pokemon-classification`
- **Alternative**: [Kaggle: Pixel Art 89,000 Sprites](https://www.kaggle.com/datasets/ebrahimelgazar/pixel-art) (`ebrahimelgazar/pixel-art`)

This notebook trains a **3.0 Million Parameter Conditional DCGAN** to synthesize 128x128 RGB sprite images conditioned on elemental types (Fire, Water, Grass, Electric, Psychic, Dragon, etc.). It then applies symmetric INT8 quantization and exports a single, self-contained `kryonimage_3m.krai` binary with a 64-byte `ModelHeader` ready to load directly on KryonOS.

## 1. Install Dependencies & Verify GPU

In [ ]:
!pip install -q torch torchvision datasets pillow numpy kagglehub

import os
import math
import struct
import json
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Hardware Accelerator: {DEVICE}")
if torch.cuda.is_available():
    print(f"[+] GPU Model: {torch.cuda.get_device_name(0)}")

## 2. Hyperparameters & Architectural Configuration

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = 128        # 128x128 RGB output (RGB565 DMA format)
LATENT_DIM = 64         # Latent vector z in R^64
NUM_CLASSES = 18        # 18 Elemental / Sprite Types
CLASS_EMBED_DIM = 64    # Class embedding dimension
EPOCHS = 40             # Fast training on Colab T4 GPU (~12-15 mins)
LEARNING_RATE_G = 2e-4
LEARNING_RATE_D = 2e-4
BETA1 = 0.5
BETA2 = 0.999

KRAI_MAGIC = 0x4B524149 # "KRAI" (0x4B524149)
MODEL_TYPE_IMAGE = 1    # MODEL_TYPE_IMAGE_GAN_VAE
QUANT_INT8 = 1

ELEMENTAL_CLASSES = [
    "Normal", "Fire", "Water", "Grass", "Electric", "Ice",
    "Fighting", "Poison", "Ground", "Flying", "Psychic", "Bug",
    "Rock", "Ghost", "Dragon", "Steel", "Fairy", "Dark"
]

print(f"[+] Elemental Classes ({len(ELEMENTAL_CLASSES)}): {', '.join(ELEMENTAL_CLASSES)}")

## 3. Dataset Loading & Preprocessing

In [ ]:
class PokemonSpriteDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(path).convert("RGBA")
            canvas = Image.new("RGBA", img.size, (255, 255, 255, 255))
            canvas.paste(img, (0, 0), img)
            img = canvas.convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (255, 255, 255))
        
        if self.transform:
            img = self.transform(img)
        return img, label

transform_pipeline = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

data_dir = "./dataset_pokemon"
os.makedirs(data_dir, exist_ok=True)
image_paths = []
labels = []

try:
    from datasets import load_dataset
    print("[+] Streaming sprite dataset from Hugging Face Hub (reach-vb/pokemon-blip-captions)...")
    hf_ds = load_dataset("reach-vb/pokemon-blip-captions", split="train")
    for i, item in enumerate(hf_ds):
        img_path = os.path.join(data_dir, f"sprite_{i:04d}.png")
        if not os.path.exists(img_path):
            raw_img = item["image"].convert("RGB")
            raw_img.save(img_path)
        image_paths.append(img_path)
        caption = str(item.get("text", "")).lower()
        assigned_label = 0
        for type_idx, type_name in enumerate(ELEMENTAL_CLASSES):
            if type_name.lower() in caption:
                assigned_label = type_idx
                break
        else:
            assigned_label = i % len(ELEMENTAL_CLASSES)
        labels.append(assigned_label)
    print(f"[+] Ingested {len(image_paths)} labeled sprites.")
except Exception as e:
    print(f"[!] Hugging Face notice: {e}")

if len(image_paths) == 0:
    print("[+] Synthesizing fallback high-contrast pixel art dataset...")
    for i in range(500):
        img_path = os.path.join(data_dir, f"synth_{i:04d}.png")
        if not os.path.exists(img_path):
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (255, 255, 255))
            pixels = img.load()
            c_r, c_g, c_b = (i * 37) % 256, (i * 67) % 256, (i * 97) % 256
            center = IMAGE_SIZE // 2
            for y in range(20, IMAGE_SIZE - 20):
                span = int(math.sin(y * 0.1) * 20 + 25)
                for x in range(center - span, center + span):
                    if (x + y + i) % 7 != 0:
                        pixels[x, y] = (c_r, c_g, c_b)
            img.save(img_path)
        image_paths.append(img_path)
        labels.append(i % len(ELEMENTAL_CLASSES))

train_dataset = PokemonSpriteDataset(image_paths, labels, transform=transform_pipeline)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"[+] DataLoader initialized with {len(train_dataset)} samples.")

## 4. Define 3.0M Parameter Conditional Generator & Discriminator

In [ ]:
class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=64, num_classes=18, embed_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.class_embed = nn.Embedding(num_classes, embed_dim)
        in_dim = latent_dim + embed_dim

        self.init_proj = nn.Sequential(
            nn.Linear(in_dim, 8 * 8 * 384, bias=False),
            nn.BatchNorm1d(8 * 8 * 384),
            nn.ReLU(True)
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(384, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True)
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True)
        )
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True)
        )
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True)
        )
        self.out_conv = nn.Sequential(
            nn.Conv2d(32, 3, 3, 1, 1, bias=True),
            nn.Tanh()
        )

    def forward(self, z, labels):
        c = self.class_embed(labels)
        x = torch.cat([z, c], dim=1)
        x = self.init_proj(x)
        x = x.view(-1, 384, 8, 8)
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x)
        out = self.out_conv(x)
        return out

class ConditionalDiscriminator(nn.Module):
    def __init__(self, num_classes=18, embed_dim=64):
        super().__init__()
        self.class_embed = nn.Embedding(num_classes, 128 * 128)
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 8, 1, 0, bias=False)
        )

    def forward(self, img, labels):
        c = self.class_embed(labels).view(-1, 1, 128, 128)
        x = torch.cat([img, c], dim=1)
        out = self.conv(x)
        return out.view(-1, 1).squeeze(1)

netG = ConditionalGenerator(LATENT_DIM, NUM_CLASSES, CLASS_EMBED_DIM).to(DEVICE)
netD = ConditionalDiscriminator(NUM_CLASSES, CLASS_EMBED_DIM).to(DEVICE)

g_params = sum(p.numel() for p in netG.parameters())
d_params = sum(p.numel() for p in netD.parameters())
print(f"[+] Generator Parameters:     {g_params:,} (~{g_params/1e6:.2f}M Params)")
print(f"[+] Discriminator Parameters: {d_params:,} (~{d_params/1e6:.2f}M Params)")

## 5. Adversarial Training Loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizerG = torch.optim.Adam(netG.parameters(), lr=LEARNING_RATE_G, betas=(BETA1, BETA2))
optimizerD = torch.optim.Adam(netD.parameters(), lr=LEARNING_RATE_D, betas=(BETA1, BETA2))

step = 0
for epoch in range(1, EPOCHS + 1):
    for real_imgs, real_labels in train_loader:
        b_size = real_imgs.size(0)
        real_imgs = real_imgs.to(DEVICE)
        real_labels = real_labels.to(DEVICE)

        # Train Discriminator
        netD.zero_grad()
        label_real = torch.full((b_size,), 0.9, device=DEVICE)
        output_real = netD(real_imgs, real_labels)
        errD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, LATENT_DIM, device=DEVICE)
        fake_labels = torch.randint(0, NUM_CLASSES, (b_size,), device=DEVICE)
        fake_imgs = netG(noise, fake_labels)
        
        label_fake = torch.full((b_size,), 0.0, device=DEVICE)
        output_fake = netD(fake_imgs.detach(), fake_labels)
        errD_fake = criterion(output_fake, label_fake)

        errD = errD_real + errD_fake
        errD.backward()
        optimizerD.step()

        # Train Generator
        netG.zero_grad()
        output_g = netD(fake_imgs, fake_labels)
        errG = criterion(output_g, torch.full((b_size,), 1.0, device=DEVICE))
        errG.backward()
        optimizerG.step()

        step += 1
        if step % 25 == 0 or step == 1:
            print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Step {step:04d} | Loss D: {errD.item():.4f} | Loss G: {errG.item():.4f}")

print("[+] Training Finished Successfully!")

## 6. Preview Generated Sprites

In [ ]:
from IPython.display import display

netG.eval()
with torch.no_grad():
    sample_noise = torch.randn(8, LATENT_DIM, device=DEVICE)
    sample_labels = torch.tensor([1, 2, 3, 4, 10, 13, 14, 17], device=DEVICE)
    generated = netG(sample_noise, sample_labels).cpu()
    generated = (generated * 0.5 + 0.5).clamp(0, 1)
    
    grid_img = Image.new("RGB", (IMAGE_SIZE * 4, IMAGE_SIZE * 2))
    for i in range(8):
        row = i // 4
        col = i % 4
        t_img = transforms.ToPILImage()(generated[i])
        grid_img.paste(t_img, (col * IMAGE_SIZE, row * IMAGE_SIZE))
    
    grid_img.save("preview_sprites.png")
    display(grid_img)

## 7. Quantize to INT8 & Export `.krai` Binary

In [ ]:
netG.eval()
weights_bytes = bytearray()

with torch.no_grad():
    for name, param in netG.named_parameters():
        arr = param.detach().cpu().numpy().astype(np.float32)
        max_val = float(np.max(np.abs(arr)))
        scale = max_val / 127.0 if max_val > 0 else 1.0
        quantized = np.clip(np.round(arr / scale), -128, 127).astype(np.int8)
        
        name_bytes = name.encode('utf-8')
        shape = list(quantized.shape)
        tensor_hdr = struct.pack(f"<B{len(name_bytes)}sfB", len(name_bytes), name_bytes, scale, len(shape))
        for d in shape:
            tensor_hdr += struct.pack("<I", d)
        weights_bytes.extend(tensor_hdr)
        weights_bytes.extend(quantized.tobytes())

pad_len = (16 - (len(weights_bytes) % 16)) % 16
weights_bytes.extend(b'\x00' * pad_len)

vocab_bytes = bytearray()
for cls_idx, cls_name in enumerate(ELEMENTAL_CLASSES):
    encoded = cls_name.encode('utf-8')
    vocab_bytes.extend(struct.pack(f"<HH{len(encoded)}s", cls_idx, len(encoded), encoded))

vocab_pad = (16 - (len(vocab_bytes) % 16)) % 16
vocab_bytes.extend(b'\x00' * vocab_pad)

HEADER_SIZE = 64
vocab_offset = HEADER_SIZE
weights_offset = vocab_offset + len(vocab_bytes)
weights_size = len(weights_bytes)

header = struct.pack(
    "<IHHIIHHHHHHIIIHH20s",
    KRAI_MAGIC,
    1,
    MODEL_TYPE_IMAGE,
    g_params,
    QUANT_INT8,
    LATENT_DIM,
    384,
    5,
    1,
    NUM_CLASSES,
    LATENT_DIM,
    vocab_offset,
    weights_offset,
    weights_size,
    IMAGE_SIZE,
    IMAGE_SIZE,
    b'\x00' * 20
)

output_file = "kryonimage_3m.krai"
with open(output_file, "wb") as f:
    f.write(header)
    f.write(vocab_bytes)
    f.write(weights_bytes)

size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"[+] Generated Model: {output_file} ({size_mb:.2f} MB)")
print("[+] Ready for ESP32-S3 deployment on KryonOS!")

## 8. Download `.krai` Model to Computer

In [ ]:
try:
    from google.colab import files
    files.download("kryonimage_3m.krai")
    print("[+] Initiated browser download for kryonimage_3m.krai!")
except Exception as e:
    print(f"[!] Download manual path: {os.path.abspath('kryonimage_3m.krai')}")